# 05. Sell-state dataset from conditional-buy OOF

조건부 매수 모델이 과거 날짜를 보지 않고 생성한 OOF 후보만으로 매도 상태를 만든다.

- 진입: `first_ask[t+1]`
- 첫 매도 판단: 진입한 1분봉이 완성된 직후
- 판단 feature: 그때까지 완성된 30봉
- 매도 체결 proxy: 판단 다음 분의 `first_bid`
- 최대 5분 보유, 5번째 상태는 강제 EXIT
- optimal action: 지금 EXIT와 미래 5분 내 최고 EXIT를 비교하며 0.2% 이내면 즉시 EXIT

In [1]:
from pathlib import Path
import json
import math
import time

import numpy as np
import pandas as pd


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'AGENT.md').is_file() and (candidate / 'README.md').is_file():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')


PROJECT_ROOT = find_project_root()
PREPROCESS_ROOT = (PROJECT_ROOT / '../../results/preprocessing').resolve()
MODELING_ROOT = (PROJECT_ROOT / '../../results/modeling').resolve()
RAW_ROOT = (PROJECT_ROOT / '../../data/stock_data/raw').resolve()
PREPROCESS_VERSION = 'scalp_30m_ohlcv_net3_minbid_5m_v1'
BUY_VERSION = 'conditional_buy_mptsnet_v1'
SELL_DATA_VERSION = 'sell_states_conditional_buy_oof_v1'
MAX_HOLD_MINUTES = 5
EXIT_MARGIN = 0.002
STAKE_USD = 100.0

schema = json.loads((PREPROCESS_ROOT / f'{PREPROCESS_VERSION}_schema.json').read_text())
metadata = pd.read_parquet(PREPROCESS_ROOT / f'{PREPROCESS_VERSION}_metadata.parquet')
with np.load(PREPROCESS_ROOT / f'{PREPROCESS_VERSION}_features.npz') as loaded:
    all_features = loaded['sequence']
default_indices = [schema['feature_names'].index(name) for name in schema['default_feature_names']]
X = np.ascontiguousarray(all_features[:, :, default_indices], dtype=np.float32)
del all_features
buy_oof = pd.read_parquet(MODELING_ROOT / f'{BUY_VERSION}_oof_predictions.parquet')
buy_test = pd.read_parquet(MODELING_ROOT / f'{BUY_VERSION}_test_predictions.parquet')

costs = schema['execution_costs']
SECTION31_RATE = float(costs['section31_rate_per_sold_dollar'])
TAF_PER_SHARE = float(costs['finra_taf_per_sold_share'])
TAF_CAP = float(costs['finra_taf_cap_usd'])
COMMISSION_PER_SIDE = float(costs['commission_usd_per_side'])

def executable_net_return(entry_ask, exit_bid):
    shares = STAKE_USD / entry_ask
    exit_notional = shares * exit_bid
    section31 = exit_notional * SECTION31_RATE
    taf = min(shares * TAF_PER_SHARE, TAF_CAP)
    costs_usd = 2 * COMMISSION_PER_SIDE + section31 + taf
    return (exit_notional - STAKE_USD - costs_usd) / STAKE_USD

print('OOF candidates:', int(buy_oof['sell_training_candidate'].sum()))
print('Test raw signals:', int(buy_test['buy_signal'].sum()))

OOF candidates: 4072
Test raw signals: 551


## Quote와 feature state 정렬

OOF 학습 진입은 동일 종목에서 고정 5분 간격으로 추려 인접 중복을 줄인다. Test는 실제 sequential simulator가 포지션 상태에 따라 진입을 거르므로 모든 raw buy signal의 경로를 보존한다.

In [2]:
def fixed_cooldown_entries(frame, signal_column, minutes=5):
    candidates = frame[frame[signal_column]].sort_values('entry_timestamp')
    selected = []
    for _, group in candidates.groupby(['session', 'symbol'], sort=False):
        available_at = None
        for index, row in group.iterrows():
            if available_at is None or row['entry_timestamp'] >= available_at:
                selected.append(index)
                available_at = row['entry_timestamp'] + pd.Timedelta(minutes=minutes)
    return frame.loc[selected].sort_values('entry_timestamp').copy()


oof_entries = fixed_cooldown_entries(buy_oof, 'sell_training_candidate', MAX_HOLD_MINUTES)
test_entries = buy_test[buy_test['buy_signal']].sort_values('entry_timestamp').copy()
oof_entries['dataset_split'] = 'oof'
test_entries['dataset_split'] = 'test'
entries = pd.concat([oof_entries, test_entries], ignore_index=True)

quote_frames = []
for path in sorted(RAW_ROOT.rglob('*_enriched.csv')):
    quote = pd.read_csv(path, usecols=['symbol', 'timestamp_utc', 'first_bid'])
    quote['timestamp_utc'] = pd.to_datetime(quote['timestamp_utc'], utc=True, errors='coerce')
    quote['symbol'] = quote['symbol'].astype(str).str.upper()
    quote['session'] = path.parent.name
    quote_frames.append(quote)
quotes = pd.concat(quote_frames, ignore_index=True)
quotes['first_bid'] = pd.to_numeric(quotes['first_bid'], errors='coerce')
quotes = quotes.drop_duplicates(['session', 'symbol', 'timestamp_utc'], keep='last')
quote_lookup = quotes.set_index(['session', 'symbol', 'timestamp_utc'])['first_bid']

state_lookup_frame = metadata[[
    'session', 'symbol', 'decision_bar_timestamp', 'feature_index',
]].copy()
state_lookup_frame['symbol'] = state_lookup_frame['symbol'].astype(str).str.upper()
assert not state_lookup_frame.duplicated(['session', 'symbol', 'decision_bar_timestamp']).any()
state_lookup = state_lookup_frame.set_index(['session', 'symbol', 'decision_bar_timestamp'])['feature_index']

print('OOF entries after fixed cooldown:', len(oof_entries))
print('Test hypothetical entries:', len(test_entries))
print('quote rows:', len(quotes))

OOF entries after fixed cooldown: 1296
Test hypothetical entries: 551
quote rows: 127423


## Optimal-stopping target 생성

각 진입에서 1~5분 뒤 next-minute first bid 수익 경로를 만든다. 현재 EXIT보다 미래 최고 EXIT가 0.2% 넘게 좋을 때만 HOLD가 정답이다. future downside는 현재 EXIT 대비 앞으로 발생할 수 있는 최저 수익 변화다.

In [3]:
CONTEXT_NAMES = [
    'current_net_return', 'max_net_return_so_far', 'min_net_return_so_far',
    'drawdown_from_peak', 'elapsed_fraction', 'buy_score', 'buy_tp_probability',
    'buy_predicted_timeout_no_tp', 'buy_predicted_downside_no_tp',
]
sequence_states, context_states, records = [], [], []
quality = {'entries_total': len(entries), 'entries_valid': 0, 'missing_state': 0, 'missing_quote': 0}
started = time.time()

for entry_number, entry in entries.iterrows():
    session = entry['session']
    symbol = str(entry['symbol']).upper()
    entry_timestamp = pd.Timestamp(entry['entry_timestamp'])
    entry_ask = float(entry['entry_ask'])
    state_indices, exit_timestamps, exit_bids = [], [], []
    valid = True
    for minute in range(1, MAX_HOLD_MINUTES + 1):
        decision_bar_timestamp = entry_timestamp + pd.Timedelta(minutes=minute - 1)
        exit_timestamp = entry_timestamp + pd.Timedelta(minutes=minute)
        state_key = (session, symbol, decision_bar_timestamp)
        quote_key = (session, symbol, exit_timestamp)
        if state_key not in state_lookup.index:
            quality['missing_state'] += 1
            valid = False
            break
        if quote_key not in quote_lookup.index:
            quality['missing_quote'] += 1
            valid = False
            break
        exit_bid = float(quote_lookup.loc[quote_key])
        if not np.isfinite(exit_bid) or exit_bid <= 0:
            quality['missing_quote'] += 1
            valid = False
            break
        state_indices.append(int(state_lookup.loc[state_key]))
        exit_timestamps.append(exit_timestamp)
        exit_bids.append(exit_bid)
    if not valid:
        continue
    path_returns = np.asarray([executable_net_return(entry_ask, bid) for bid in exit_bids], dtype=np.float32)
    if not np.isfinite(path_returns).all():
        quality['missing_quote'] += 1
        continue
    quality['entries_valid'] += 1
    for minute in range(1, MAX_HOLD_MINUTES + 1):
        position = minute - 1
        current_return = float(path_returns[position])
        past_returns = path_returns[:position + 1]
        if minute < MAX_HOLD_MINUTES:
            best_later_return = float(path_returns[position + 1:].max())
            worst_later_return = float(path_returns[position + 1:].min())
            hold_advantage = best_later_return - current_return
            future_downside_from_now = worst_later_return - current_return
            optimal_hold = int(hold_advantage > EXIT_MARGIN)
            oracle_exit_minute = int(position + 2 + np.argmax(path_returns[position + 1:]))
        else:
            best_later_return = current_return
            worst_later_return = current_return
            hold_advantage = 0.0
            future_downside_from_now = 0.0
            optimal_hold = 0
            oracle_exit_minute = MAX_HOLD_MINUTES
        context = np.asarray([
            current_return, float(past_returns.max()), float(past_returns.min()),
            current_return - float(past_returns.max()), minute / MAX_HOLD_MINUTES,
            float(entry['buy_score']), float(entry['tp_probability']),
            float(entry['predicted_timeout_no_tp']), float(entry['predicted_downside_no_tp']),
        ], dtype=np.float32)
        state_index = len(records)
        sequence_states.append(X[state_indices[position]].copy())
        context_states.append(context)
        records.append({
            'state_index': state_index, 'entry_id': f"{entry['dataset_split']}_{entry_number:08d}",
            'dataset_split': entry['dataset_split'], 'session': session, 'symbol': symbol,
            'entry_timestamp': entry_timestamp, 'entry_ask': entry_ask,
            'decision_bar_timestamp': entry_timestamp + pd.Timedelta(minutes=minute - 1),
            'decision_available_timestamp': exit_timestamps[position],
            'exit_execution_timestamp': exit_timestamps[position], 'exit_first_bid': exit_bids[position],
            'hold_minute': minute, 'current_net_return': current_return,
            'best_later_return': best_later_return, 'worst_later_return': worst_later_return,
            'hold_advantage': hold_advantage, 'future_downside_from_now': future_downside_from_now,
            'optimal_hold': optimal_hold, 'oracle_exit_minute': oracle_exit_minute,
            'buy_score': float(entry['buy_score']), 'buy_tp_probability': float(entry['tp_probability']),
        })

sequence_array = np.stack(sequence_states).astype(np.float32)
context_array = np.stack(context_states).astype(np.float32)
state_metadata = pd.DataFrame(records)
assert np.isfinite(sequence_array).all() and np.isfinite(context_array).all()
assert np.array_equal(state_metadata['state_index'].to_numpy(), np.arange(len(state_metadata)))
assert (state_metadata['decision_available_timestamp'] == state_metadata['exit_execution_timestamp']).all()
print(f'elapsed: {(time.time() - started) / 60:.2f} min')
print('sequence/context:', sequence_array.shape, context_array.shape)
print('quality:', quality)
display(state_metadata.groupby(['dataset_split', 'session']).agg(
    entries=('entry_id', 'nunique'), states=('state_index', 'size'), hold_rate=('optimal_hold', 'mean'),
))

elapsed: 0.02 min
sequence/context: (9065, 30, 38) (9065, 9)
quality: {'entries_total': 1847, 'entries_valid': 1813, 'missing_state': 34, 'missing_quote': 0}


entries  states  hold_rate
dataset_split session                                       
oof           session_2026-07-10      324    1620   0.430247
              session_2026-07-13      215    1075   0.459535
              session_2026-07-14      226    1130   0.431858
              session_2026-07-15      330    1650   0.439394
              session_2026-07-16      184     920   0.419565
test          session_2026-07-17      534    2670   0.404120

In [4]:
feature_path = PREPROCESS_ROOT / f'{SELL_DATA_VERSION}_features.npz'
metadata_path = PREPROCESS_ROOT / f'{SELL_DATA_VERSION}_metadata.parquet'
schema_path = PREPROCESS_ROOT / f'{SELL_DATA_VERSION}_schema.json'
np.savez_compressed(feature_path, sequence=sequence_array, context=context_array)
state_metadata.to_parquet(metadata_path, index=False, compression='zstd')
sell_schema = {
    'version': SELL_DATA_VERSION, 'buy_model_version': BUY_VERSION,
    'sequence_length': int(sequence_array.shape[1]),
    'sequence_feature_names': schema['default_feature_names'], 'context_names': CONTEXT_NAMES,
    'max_hold_minutes': MAX_HOLD_MINUTES, 'exit_margin': EXIT_MARGIN,
    'timing': {
        'state': 'completed bar ending at decision_bar_timestamp + 1 minute',
        'exit_execution': 'first_bid in the next minute at decision_available_timestamp',
    },
    'target': {
        'classification': 'optimal_hold = best later exit exceeds current exit by more than 0.2%',
        'regression': ['hold_advantage', 'future_downside_from_now'],
    },
    'quality': quality, 'artifacts': {'features': str(feature_path), 'metadata': str(metadata_path)},
}
schema_path.write_text(json.dumps(sell_schema, ensure_ascii=False, indent=2), encoding='utf-8')
print(feature_path)
print(metadata_path)

/home/user/urbandatalab/YSLee/results/preprocessing/sell_states_conditional_buy_oof_v1_features.npz
/home/user/urbandatalab/YSLee/results/preprocessing/sell_states_conditional_buy_oof_v1_metadata.parquet
